### `etd_rk43.ipynb`
*Created: September 25, 2026* <br/>
This notebook implements ETD-RK4 with adaptive step sizing, using a 4(3) embedded ETD-RK pair. 

In [1]:
using NBInclude, LinearAlgebra
@nbinclude("etd_rk_core.ipynb")
@nbinclude("../../../phi_functions/phi_functions.ipynb")

In [2]:
function compute_initial_dt(tspan::NTuple{2,Float64})
    #Fill me in...
    return 1.0
end 

compute_initial_dt (generic function with 1 method)

In [45]:
function etd_rk_step!(cache::ETDRK43Cache, prob::SemilinearODEProblem, saveat, 
                      t_current::Float64, u_current, dt::Float64, dt_min::Float64, 
                      dt_max::Float64, abstol::Float64, reltol::Float64)

    safety_factor = 0.9
    
    #We have not yet computed an accepted step
    accepted = false 

    #Track number of step attempts 
    current_attempt = 1 
    max_attempts = 1            
    
    while accepted == false && current_attempt ≤ max_attempts

        #Compute E, G, P1, P2, P3, Q1, Q2, Q3 
        compute_phi_matrices!(cache, prob, t_current, dt)

        #Compute K1, F1, K2, F2, K3, F3, K4, F4
        compute_vectors!(cache, prob, t_current, dt, u_current)
        
        #Compute B1, B2, B3, B4 
        compute_coefficients!(cache, prob, t_current, dt)

        #Compute proposed next step
        u_proposed = compute_candidate!(cache, prob, t_current, dt, u_current)

        #Compute the error estimate
        error_estimate = compute_error_estimate!(cache, dt)

        # #Decide whether to accept or reject the step 
        # step_result = process_step!(error_estimate, dt, dt_min, dt_max, abstol, reltol, safety_factor)

        # #Adjust step size based on step result and error estimate
        # update_dt!(dt, step_result, error_estimate)

        # #Whether or not the step is accepted
        # accepted = step_result == :accept ? true : false 

        current_attempt += 1 
    end

    println("finished with this step.")

    t_next = t_current
    u_proposed = 2.0 .* u_current
    
    if accepted == true 
        return t_next, u_proposed 
    else
        error("`dt` became too small. Integration terminated.")
    end 
    
end 

etd_rk_step! (generic function with 1 method)

In [48]:
function compute_phi_matrices!(cache, prob, t, dt)    
    #Compute E, G, P1, P2, P3, Q1, Q2, Q3 

    @unpack A = prob 
    @unpack M, N, E, G, P, Q = cache 
   
    M .= dt*A
    N .= 0.5*dt*A

    E .= phi(M, 0)
    G .= phi(N, 0)

    P1, P2, P3 = P

    P1 .= phi(M, 1)
    P2 .= phi(M, 2)
    P3 .= phi(M, 3)

    Q1, Q2, Q3 = Q 

    Q1 .= phi(N, 1)
    Q2 .= phi(N, 2)
    Q3 .= phi(N, 3)
  
    return nothing
    
end 

function compute_vectors!(cache, prob, t, dt, u)
     #Compute K1, F1, K2, F2, K3, F3, K4, F4

    #u is u_n, the current solution iterate

    @unpack E, G, Q, K, F = cache
    @unpack f, p = prob

    K[1] .= u
    F[1] .= f(K[1], p, t)
    
    K[2] .= G*u + (dt/2)*Q[1]*F[1]
    F[2] .= f(K[2], p, t + dt/2)

    K[3] .= G*u + (dt/2)*((Q[1] - Q[2])*F[1] + Q[2]*F[2])
    F[3] .= f(K[3], p, t + dt/2)

    K[4] .= E*u + dt*((P[1] - 2*P[2])*F[1] + 2*P[2]*F[3]) 
    F[4] .= f(K[4], p, t + dt)

    return nothing 
end 

function compute_coefficients!(cache, prob, t, dt)
    #Compute B1, B2, B3, B4 

    @unpack P, B = cache

    B[1] .=   P[1] - 3*P[2] + 4*P[3]
    B[2] .= 2*P[2] - 4*P[3]
    B[3] .= 2*P[2] - 4*P[3]
    B[4] .=  -P[2] + 4*P[3]    
end 

function compute_candidate!(cache, prob, t, dt, u_current)

    @unpack E, K, F, B = cache
    p = prob.p
    
    K[5] .= E*u_current + dt*(B[1]*F[1] + B[2]*F[2] + B[3]*F[3] + B[4]*F[4])
    F[5] .= f(K[5], p, t + dt)
    
    cache.u_candidate .= K[5]    
end 


function compute_error_estimate!(cache, dt)
    @unpack B, F = cache
    cache.error_estimate = dt*B[4]*(F[5] .- F[4])
end 

function process_step!(error_estimate, dt, dt_min, dt_max, abstol, reltol, safety_factor)
    #Compute B1, B2, B3, B4 
end 
      
        
function update_dt!(dt, step_result, error_estimate)
    #Adjust step size based on step result and error estimate
end

update_dt! (generic function with 1 method)

In [33]:
function solve(prob::SemilinearODEProblem; saveat = nothing, max_steps::Int = 1000, 
               abstol::Float64 = 1e-6, reltol::Float64 = 1e-3, 
               initial_dt = nothing, dt_min::Float64 = 1e-8, dt_max = 10_000.0) 

    """
    Solve the semilinear ODE u_t = Au + f(u,p,t), u(t0) = u0 on the time interval [t0, tf]. 
    
    PARAMTERS
    ---------
    prob :: a SemilinearODEProblem
    initial_dt :: the
    saveat :: times at which to save the solution
    initial_dt :: initial time step 
    max_steps :: maximum number of time steps 
    abstol :: absolute tolerance for the adaptive step-sizing algorithm 
    reltol :: relative tolerance for the adaptive step-sizing algorithm 

    RETURNS
    -------
    sol.u ::
    sol.t ::     
    sol.p :: 
    """

    @unpack A, f, u0, tspan, p = prob 
    t0, tf = tspan

    #Create array to store the accepted time steps 
    t = Float64[t0]
    sizehint!(t, max_steps)

    #Ensure u0 is a float or vector of floats
    u0 = float.(u0)
    u = [u0]
    sizehint!(u, max_steps)
   
    #Initialize time step
    dt = initial_dt == nothing ? compute_initial_dt(tspan) : initial_dt 

    println("initial_dt = $(dt).")
    
    t_current = t[1]
    u_current = u[1]

    cache = ETDRK43Cache(u0)
  
    current_step = 1

    println("Starting integration loop...")
  
    # while t_current < tf && current_step ≤ max_steps

    #     t_current = t[end]
    #     u_current = u[end]
        
    #     t_next, u_next = etd_rk_step!(cache, prob, saveat, t_current, u_current, dt, dt_min, dt_max, abstol, reltol)

    #     @show t_next
    #     @show u_next

        
        
    #     push!(u, u_next)
    #     push!(t, t_next)

    #     current_step += 1 
    # end 

  
    return (u = u, t = t, prob = prob, steps = current_step)
end

solve (generic function with 1 method)

In [46]:
#Van der Pol oscillator
p = 10.0
A_vdp = [0.0 1.0; -1.0 p] 
vdp_nonlin(u,p,t) = [0.0, -p*(u[1])^2 * u[2]]
u0 = [2.0, 0.0]
tspan = (0.0, 50.0)

cache = ETDRK43Cache(u0);
prob = SemilinearODEProblem(A_vdp, vdp_nonlin, u0, tspan, p)
saveat = nothing
t_current = 1.0
u_current = copy(u0)
dt = 0.1
dt_min = 1e-10
dt_max = 100.0
abstol = 1e-6
reltol = 1e-3

0.001

In [47]:
etd_rk_step!(cache, prob, saveat, t_current, u_current, dt, dt_min, dt_max, abstol, reltol)    

LoadError: type ETDRK43Cache has no field u_third_order